In [1]:
import numpy as np
import pandas as pd
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer

from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
import hdbscan.validity as dbcv_module

In [3]:
df = pd.read_parquet("data/posts_preprocessed.parquet")
embeddings = np.load("data/embeddings_USER-bge-m3_300k.npy")

In [4]:
print(f"Постов: {len(df)}, Эмбеддингов: {len(embeddings)}")
assert len(df) == len(embeddings), "Размеры не совпадают!"

Постов: 300000, Эмбеддингов: 300000


In [5]:
umap_model = UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=150,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    analyzer="word", ngram_range=(1, 2), min_df=10, max_df=0.85, max_features=30_000
)

In [6]:
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    top_n_words=10,
    verbose=True,
)

docs = df["text_lemm"].tolist()

topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)

df["topic"] = topics
print(f"Тем найдено: {topic_model.get_topic_info().shape[0] - 1}")
print(f"Noise Ratio: {(np.array(topics) == -1).mean():.3f}")

2026-04-02 17:11:10,936 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-02 17:17:21,832 - BERTopic - Dimensionality - Completed ✓
2026-04-02 17:17:21,838 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-02 17:17:47,859 - BERTopic - Cluster - Completed ✓
2026-04-02 17:17:47,891 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-02 17:18:20,073 - BERTopic - Representation - Completed ✓


Тем найдено: 212
Noise Ratio: 0.445


In [7]:
# --- C_V Coherence ---
topic_words = [
    [word for word, _ in topic_model.get_topic(t)]
    for t in topic_model.get_topics()
    if t != -1
]

tokenized_docs = [doc.split() for doc in docs]
dictionary = Dictionary(tokenized_docs)

cm = CoherenceModel(
    topics=topic_words, texts=tokenized_docs, dictionary=dictionary, coherence="c_v"
)
coherence_cv = cm.get_coherence()
print(f"C_V Coherence: {coherence_cv:.4f}")

C_V Coherence: 0.8083


In [8]:
# --- Topic Diversity ---
all_words = [word for words in topic_words for word in words]
unique_words = set(all_words)
diversity = len(unique_words) / len(all_words)
print(f"Topic Diversity: {diversity:.4f}")

Topic Diversity: 0.7736


In [9]:
# --- DBCV ---
reduced = topic_model.umap_model.embedding_
labels = np.array(topics)

mask = labels != -1
dbcv_score = dbcv_module.validity_index(reduced[mask].astype(np.float64), labels[mask])
print(f"DBCV: {dbcv_score:.4f}")

DBCV: 0.3889


In [11]:
metrics = {
    "n_topics": topic_model.get_topic_info().shape[0] - 1,
    "noise_ratio": float((np.array(topics) == -1).mean()),
    "coherence_cv": float(coherence_cv),
    "topic_diversity": float(diversity),
    "dbcv": float(dbcv_score),
}

print("\n--- BASELINE METRICS ---")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

topic_model.save("models/bertopic_baseline")
df.to_parquet("data/posts_with_topics.parquet", index=False)

2026-04-02 17:26:25,298 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.



--- BASELINE METRICS ---
  n_topics: 212
  noise_ratio: 0.4447
  coherence_cv: 0.8083
  topic_diversity: 0.7736
  dbcv: 0.3889
